# 🎙️ PHILIA — Audio Emotion Fine-Tuning
**Model:** `facebook/wav2vec2-base` → fine-tuned on MELD audio  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Output:** saved to Google Drive → download and drop into PHILIA

**Runtime:** Set to `GPU` → Runtime > Change runtime type > T4 GPU

In [2]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

CUDA available: True
GPU: Tesla T4


In [3]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate soundfile librosa

In [4]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/audio_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model will be saved to: /content/drive/MyDrive/PHILIA/models/audio_emotion


In [5]:
# ── Cell 4: Download MELD dataset ──────────────────────────────────────────────
# Downloads the full MELD raw archive (~1.3 GB — takes ~3 min on Colab)
# Contains: train/dev/test mp4 clips + CSV label files
import os

MELD_DIR = '/content/MELD'
os.makedirs(MELD_DIR, exist_ok=True)

if not os.path.exists('/content/MELD.Raw.tar.gz'):
    print('Downloading MELD raw data...')
    !wget -q --show-progress http://web.eecs.umich.edu/~mihalcea/downloads/MELD.Raw.tar.gz -O /content/MELD.Raw.tar.gz
    print('Extracting...')
    !tar -xzf /content/MELD.Raw.tar.gz -C /content/MELD --strip-components=1
    print('Done.')
else:
    print('MELD already downloaded.')

!ls /content/MELD/

MELD already downloaded.
dev_sent_emo.csv	     README.txt		 train_splits
dev_splits_complete	     test_sent_emo.csv	 train.tar.gz
dev.tar.gz		     test.tar.gz
output_repeated_splits_test  train_sent_emo.csv


In [ ]:
!pip install optuna


In [ ]:

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score



In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }



In [ ]:

def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 6),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 0.1, log=True),
    }



In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    gradient_accumulation_steps=2
)



In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)



In [ ]:

best_run = trainer.hyperparameter_search(
    direction="maximize",
    hp_space=hp_space,
    n_trials=10
)

for n, v in best_run.hyperparameters.items():
    setattr(trainer.args, n, v)

trainer.train()



In [6]:
import tarfile, os

MELD_DIR = '/content/MELD'

print('Extracting train.tar.gz...')
with tarfile.open(os.path.join(MELD_DIR, 'train.tar.gz'), 'r:gz') as tar:
    tar.extractall(path=MELD_DIR)

print('Extracting dev.tar.gz...')
with tarfile.open(os.path.join(MELD_DIR, 'dev.tar.gz'), 'r:gz') as tar:
    tar.extractall(path=MELD_DIR)

print('Extracting test.tar.gz...')
with tarfile.open(os.path.join(MELD_DIR, 'test.tar.gz'), 'r:gz') as tar:
    tar.extractall(path=MELD_DIR)

print('Done extracting all MELD sub-archives.')
!ls -R /content/MELD/ | head -n 20 # List some files to verify

Extracting train.tar.gz...


/tmp/ipykernel_64755/715450495.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=MELD_DIR)


Extracting dev.tar.gz...


/tmp/ipykernel_64755/715450495.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=MELD_DIR)


Extracting test.tar.gz...


/tmp/ipykernel_64755/715450495.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=MELD_DIR)


Done extracting all MELD sub-archives.
/content/MELD/:
dev_sent_emo.csv
dev_splits_complete
dev.tar.gz
output_repeated_splits_test
README.txt
test_sent_emo.csv
test.tar.gz
train_sent_emo.csv
train_splits
train.tar.gz

/content/MELD/dev_splits_complete:
dia0_utt0.mp4
dia0_utt1.mp4
dia100_utt0.mp4
dia101_utt0.mp4
dia102_utt0.mp4
dia102_utt1.mp4
dia103_utt0.mp4


In [7]:
# ── Cell 5: Extract audio from MP4 clips ──────────────────────────────────────
# MELD comes as .mp4 files; we extract 16kHz mono WAV using ffmpeg
import subprocess, glob, os
from tqdm.auto import tqdm

AUDIO_DIR = '/content/MELD_audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

mp4_files = glob.glob('/content/MELD/**/*.mp4', recursive=True)
print(f'Found {len(mp4_files)} mp4 files')

for mp4 in tqdm(mp4_files, desc='Extracting audio'):
    wav_out = os.path.join(AUDIO_DIR, os.path.basename(mp4).replace('.mp4', '.wav'))
    if not os.path.exists(wav_out):
        subprocess.run(
            ['ffmpeg', '-i', mp4, '-ar', '16000', '-ac', '1', '-loglevel', 'error', wav_out],
            check=False
        )

print(f'Audio files extracted: {len(glob.glob(AUDIO_DIR + "/*.wav"))}')

Found 13848 mp4 files


Extracting audio:   0%|          | 0/13848 [00:00<?, ?it/s]

Audio files extracted: 11264


In [8]:
# ── Cell 6: Load CSV labels and build dataset ──────────────────────────────────
import pandas as pd
import glob, os

# Download CSV files from MELD GitHub (labels for each utterance)
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/train_sent_emo.csv -O /content/train.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/dev_sent_emo.csv   -O /content/dev.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv  -O /content/test.csv

# MELD emotion → PHILIA canonical label
EMOTION_MAP = {
    'anger':    'angry',
    'disgust':  'disgust',
    'fear':     'fear',
    'joy':      'happy',
    'neutral':  'neutral',
    'sadness':  'sad',
    'surprise': 'surprise',
}
LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

AUDIO_DIR = '/content/MELD_audio'

def load_split(csv_path, split_name):
    df = pd.read_csv(csv_path)
    # MELD filename format: diaX_uttY.mp4
    df['wav_path'] = df.apply(
        lambda r: os.path.join(
            AUDIO_DIR,
            f'dia{r["Dialogue_ID"]}_utt{r["Utterance_ID"]}.wav'
        ), axis=1
    )
    df['canonical'] = df['Emotion'].str.lower().map(EMOTION_MAP)
    df = df[df['canonical'].notna()]           # drop any unmapped
    df = df[df['wav_path'].apply(os.path.exists)]  # only files that extracted
    df['label'] = df['canonical'].map(LABEL2ID)
    print(f'{split_name}: {len(df)} samples | dist: {dict(df["canonical"].value_counts())}')
    return df[['wav_path', 'label', 'canonical']].reset_index(drop=True)

train_df = load_split('/content/train.csv', 'TRAIN')
val_df   = load_split('/content/dev.csv',   'VAL')
test_df  = load_split('/content/test.csv',  'TEST')

TRAIN: 9989 samples | dist: {'neutral': np.int64(4710), 'happy': np.int64(1743), 'surprise': np.int64(1205), 'angry': np.int64(1109), 'sad': np.int64(683), 'disgust': np.int64(271), 'fear': np.int64(268)}
VAL: 1109 samples | dist: {'neutral': np.int64(470), 'happy': np.int64(163), 'angry': np.int64(153), 'surprise': np.int64(150), 'sad': np.int64(111), 'fear': np.int64(40), 'disgust': np.int64(22)}
TEST: 2610 samples | dist: {'neutral': np.int64(1256), 'happy': np.int64(402), 'angry': np.int64(345), 'surprise': np.int64(281), 'sad': np.int64(208), 'disgust': np.int64(68), 'fear': np.int64(50)}


In [9]:
# ── Cell 7: Create HuggingFace Dataset + feature extractor ──────────────────
import torch, librosa
import numpy as np
from datasets import Dataset
from transformers import Wav2Vec2FeatureExtractor

MODEL_CHECKPOINT = 'facebook/wav2vec2-base'
SAMPLE_RATE = 16000
MAX_DURATION = 6.0   # seconds — clips > 6s are truncated

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_CHECKPOINT)

def df_to_dataset(df):
    return Dataset.from_pandas(df[['wav_path', 'label']])

def preprocess(batch):
    audio_arrays = []
    for path in batch['wav_path']:
        arr, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True,
                              duration=MAX_DURATION)
        audio_arrays.append(arr)
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        truncation=True,
        max_length=int(SAMPLE_RATE * MAX_DURATION),
        return_tensors='np',
    )
    return {
        'input_values': inputs.input_values,
        'labels': batch['label'],
    }

print('Preprocessing train...')
train_ds = df_to_dataset(train_df).map(preprocess, batched=True, batch_size=32, remove_columns=['wav_path'])
print('Preprocessing val...')
val_ds   = df_to_dataset(val_df).map(preprocess,   batched=True, batch_size=32, remove_columns=['wav_path'])
print('Preprocessing test...')
test_ds  = df_to_dataset(test_df).map(preprocess,  batched=True, batch_size=32, remove_columns=['wav_path'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Datasets ready.')

Preprocessing train...


Map:   0%|          | 0/9989 [00:00<?, ? examples/s]

Preprocessing val...


Map:   0%|          | 0/1109 [00:00<?, ? examples/s]

Preprocessing test...


Map:   0%|          | 0/2610 [00:00<?, ? examples/s]

Datasets ready.


In [16]:
# ── Cell 8: Load model ────────────────────────────────────────────────────────
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

# Freeze feature encoder — only fine-tune the transformer layers
model.freeze_feature_encoder()

total   = sum(p.numel() for p in model.parameters())
trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total/1e6:.1f}M | Trainable: {trained/1e6:.1f}M')

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params: 94.6M | Trainable: 90.4M


In [17]:
# ── Cell 9: Training ──────────────────────────────────────────────────────────
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Optional, Union
import torch

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

@dataclass
class DataCollatorWithPadding:
    feature_extractor: Wav2Vec2FeatureExtractor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_values = [{'input_values': f['input_values']} for f in features]
        labels = torch.tensor([f['labels'] for f in features], dtype=torch.long)
        batch = self.feature_extractor.pad(input_values, padding=self.padding, return_tensors='pt')
        batch['labels'] = labels
        return batch

data_collator = DataCollatorWithPadding(feature_extractor=feature_extractor)

# Calculate total training steps for warmup_steps
num_train_samples = len(train_ds)
per_device_train_batch_size = 8
gradient_accumulation_steps = 2
num_train_epochs = 5 # Increased from 10 to 15
total_training_steps = int(num_train_samples / (per_device_train_batch_size * gradient_accumulation_steps)) * num_train_epochs
warmup_steps_calculated = int(0.1 * total_training_steps)

training_args = TrainingArguments(
    output_dir='/content/wav2vec2_audio_emotion',
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=warmup_steps_calculated,
    learning_rate=1e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print('Starting training...')
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,3.061519,1.600880,0.423805
2,3.040158,1.544196,0.455365
3,3.006188,1.555158,0.440938
4,2.818574,1.532300,0.460775
5,2.822812,1.524607,0.463481


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3125, training_loss=2.98893423828125, metrics={'train_runtime': 2033.8853, 'train_samples_per_second': 24.556, 'train_steps_per_second': 1.536, 'total_flos': 2.72063081052e+18, 'train_loss': 2.98893423828125, 'epoch': 5.0})

In [18]:
# ── Cell 10: Evaluate on test set ─────────────────────────────────────────────
results = trainer.evaluate(test_ds)
print('Test results:', results)

Test results: {'eval_loss': 1.4747673273086548, 'eval_accuracy': 0.48084291187739464, 'eval_runtime': 32.6064, 'eval_samples_per_second': 80.046, 'eval_steps_per_second': 10.029, 'epoch': 5.0}


In [19]:
# ── Cell 11: Save to Google Drive ─────────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
trainer.model.save_pretrained(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

Saving model to /content/drive/MyDrive/PHILIA/models/audio_emotion ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done! Files in Drive:
  config.json  (0.0 MB)
  model.safetensors  (378.3 MB)
  preprocessor_config.json  (0.0 MB)


In [14]:
# ── Cell 12: (Optional) Push to Hugging Face Hub ──────────────────────────────
# Only run this cell if you have a HuggingFace account
# Replace YOUR_HF_USERNAME and YOUR_HF_TOKEN below

# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-audio-emotion')
# feature_extractor.push_to_hub('YOUR_HF_USERNAME/philia-audio-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')

Skipped (optional). Uncomment above lines to push to HF Hub.
